In [68]:
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
import re

pd.set_option('display.max_colwidth', None)

In [69]:
#Ucitavanje podataka
proizvodi = pd.read_csv(r'..\data\raw\cleaned_makeup_products.csv', na_values=pd.NA)

# 1. Proizvodi - pregled

In [70]:
proizvodi.columns

Index(['product_link_id', 'product_link', 'category', 'item_id',
       'product_name', 'brand', 'price', 'num_shades', 'rating', 'num_reviews',
       'description', 'pros', 'cons', 'best_uses', 'describe_yourself',
       'review_star_1', 'review_star_2', 'review_star_3', 'review_star_4',
       'review_star_5', 'rating_star_1', 'rating_star_2', 'rating_star_3',
       'rating_star_4', 'rating_star_5', 'rating_count', 'review_count',
       'average_rating', 'recommended_ratio', 'native_review_count',
       'native_sampling_review_count', 'native_community_content_review_count',
       'syndicated_review_count', 'faceoff_negative', 'faceoff_positive'],
      dtype='str')

In [71]:
display(proizvodi.sample(random_state=42, n=2)[["product_link","item_id","product_name","description","brand","category"]])

,product_link,item_id,product_name,description,brand,category
430,https://www.ulta.com/p/supercharged-dewy-skin-primer-pimprod2021488?sku=2574046,2574046.0,Supercharged Dewy Skin Primer,"Use the following zoom and pan buttons to control the image that follows them Tab through the images or use the previous or next buttons to navigate each product image Milani Supercharged Dewy Skin Primer Item 2574046 4.6 4.6 out of 5 stars. 115 reviews 115 Reviews Ask A Question $11.99 Size: 1.0 oz Available to ship Check in-store availability See same day delivery eligibility in bag ADD TO BAG Summary Conscious Beauty at Ulta Beauty™ Vegan Clean Ingredients Cruelty Free Give Back Milani Supercharged Dewy Skin Primer instantly leaves skin looking energized while creating a radiant, glass-like canvas for makeup. Details Features Benefits Key Ingredients Wear alone or underneath your favorite complexion product to boost luminosity Vegan Made in the U.S.A. Cruelty-Free Smooths skin, extends makeup wear Leaves an ultra-dewy finish Enhanced with Citrus, Ginseng and Turmeric Extracts Squalene and Jojoba Milk help hydrate and soothe skin How To Use Use alone for a dewy glow or underneath your favorite complexion product to boost luminosity Ingredients INGREDIENTS: Water / Aqua / Eau, ?Butylene Glycol?, Glycerin, Simmondsia Chinensis (Jojoba) Seed Oil, ?Tridecyl Trimellitate?, ?Glyceryl Stearate Citrate?, Triethylhexanoin, ?Polyglyceryl-3 Stearate?, Prunus Amygdalus Dulcis (Sweet Almond) Oil, Vegetable Oil / Olus Oil/ Huile végétale, Citrus Aurantium Dulcis (Orange) Fruit Extract, Leuconostoc/Radish Root Ferment Filtrate, Panthenol, Tocopherol, ?Hydrogenated Lecithin?, ?Panax Ginseng Root Extract?, ?Squalane?, ?Sodium Hyaluronate?, ?Propanediol?, ?Phospholipids?, ?Citrus Limon (Lemon) Fruit Extract?, ?Curcuma Longa (Turmeric) Root Extract?, ?Polysorbate 60?, Sorbitan Isostearate, ?Ethylene/Propylene/Styrene Copolymer?, ?Butylene/Ethylene/Styrene Copolymer?, ?Caprylyl/Capryl Glucoside?, ?Xanthan Gum?, ?Glyceryl Undecylenate?, ?Glyceryl Caprylate?, ?Hydroxyacetophenone?, ?Hydroxyethyl Acrylate/Sodium Acryloyldimethyl Taurate Copolymer, Phenoxyethanol, Ethylhexylglycerin, ?Citrus Aurantium Dulcis (Orange) Oil?.",Ask A Question,Face Primer
588,https://www.ulta.com/p/matte-beauty-blush-wand-pimprod2043369?sku=2619515,2619515.0,Matte Beauty Blush Wand,"Use the following zoom and pan buttons to control the image that follows them try it Tab through the images or use the previous or next buttons to navigate each product image Charlotte Tilbury Matte Beauty Blush Wand Item 2619515 4.5 4.5 out of 5 stars. 795 reviews 795 Reviews $42.00 Free Gift with purchase Color: Pillow Talk Dream Pop matte cherry pink Size: 0.4 oz Available to ship Check in-store availability See same day delivery eligibility in bag ADD TO BAG Summary Matte Beauty Blush Wand is a highly pigmented matte, liquid blush for a fresh, flushed look to revive your complexion. Inspired by Charlotte Tilbury's best-selling beauty light wands, dot and blend to apply. Details Features Research Results Twist to open the sponge applicator, dot onto cheeks, blend, and don't forget to twist and close the applicator after use. This product is light-weight and blends beautifully with other makeup products. Pro Tip: Apply over cream and liquid products, before setting with powders. 97% agree it gives a FRESH GLOW of COLOUR** 92% agree it LIFTS the LOOK of your COMPLEXION** 93% agree it GLIDES ON EASILY** 81% agree it LASTS ALL DAY** ** Tested on 106 people over 1 week How To Use Dial up the dots: Charlotte's signature cushion applicator allows for a buildable application: 1 dot for a natural flush, 2-3 dots for an intensified blushing look. Blend: using the Hollywood Complexion Brush (sold separately), blend after each application up along the cheekbones towards the temples. Charlotte Tilbury Tip: The quick drying formula sets in seconds! Remember: Twist to open the sponge applicator, dot onto cheeks, blend, 

### Zapažanje
- SKU proizvoda iz URL-a odredjuje item_id
- opis sadrži nepotrebne vrednosti:
    - na pocetku: Use the following zoom and pan buttons to control the image that follows them try it Tab through the images or use the previous or next buttons to navigate each product **image** ili **...ADD TO BAG Summary**
    - na kraju: **Shipping & Coupon Restrictions**
- opis sadrži **rating** i **review_count** u trenutku scrape-a kao i **naziv**, **brend** i **sku** proizvoda
- category sadrži nevalidne vrednosti 


In [72]:
print('Broj redova:', proizvodi.shape[0], 'Broj kolona:', proizvodi.shape[1])

Broj redova: 1373 Broj kolona: 35


In [73]:
def prikazi_nedostajuce_vrednosti(tabela):
    nedostajuce = pd.DataFrame({'broj_nedostajucih_vr': tabela.isna().sum(), 'udeo_nedostajucih_vr': tabela.isna().mean()*100})
    nedostajuce = (nedostajuce[nedostajuce['broj_nedostajucih_vr'] > 0]).sort_values(by='udeo_nedostajucih_vr', ascending=False).round(2)
    display(nedostajuce)

prikazi_nedostajuce_vrednosti(proizvodi)

,broj_nedostajucih_vr,udeo_nedostajucih_vr
native_sampling_review_count,1373,100.00
num_shades,1309,95.34
describe_yourself,683,49.75
item_id,520,37.87
brand,443,32.27
rating,141,10.27
faceoff_positive,129,9.40
faceoff_negative,129,9.40
cons,116,8.45
best_uses,49,3.57


### Zapažanje

- Cena i opis proizvoda su potpuno popunjeni.
- **brand** , **rating** se može popuniti iz description.
- **item_id** se može popuniti iz product_link
- num_shade, describe_yourself polja se izostavljaju iz dalje analize

In [74]:
struktura = pd.DataFrame({
    "dtype": proizvodi.dtypes,
    "broj_jedinstvenih_vrednosti": proizvodi.nunique(dropna=True),
})

display(struktura.sort_values(by='broj_jedinstvenih_vrednosti', ascending=False))

,dtype,broj_jedinstvenih_vrednosti
product_link_id,int64,1373
product_link,str,1278
description,str,1275
product_name,str,1263
pros,str,1182
faceoff_positive,str,1094
faceoff_negative,str,1093
best_uses,str,1088
cons,str,929
item_id,float64,795


### Zapažanje

- **product_link_id** jeste identifikator pojedinačnog reda, ne proizvoda.
-  Postoje višestruko uneti proizvodi (jedinstvenih product_link_id> jedinstvenih product_link)
- Ponavljaju se proizvodi sa istim imenom, redovi verovatno predstavljaju različite varijante (jedinstvenih product_name> jedinstvenih product_link)
- Jedinstvenost izvornog reda ne dokazuje jedinstvenost stvarnog proizvoda.

# 2. Proizvod i SKU varijante

Razlikujemo izvorni red proizvoda, SKU varijantu i logički proizvod.

In [75]:
proizvodi["logical_product_link"] = proizvodi["product_link"].astype("str").str.partition("?")[0]

duplirani_proizvodi = (
    proizvodi.groupby("logical_product_link", dropna=False)
    .agg(
        broj_redova=("product_link_id", "size"),
        broj_naziva=("product_name", "nunique"),
        broj_description=("description", "nunique"),
        broj_cena=("price", "nunique"),
        broj_brendova=("brand", "nunique"),
        broj_kategorija=("category", "nunique"),
        broj_pros=("pros", "nunique"),
        broj_cons=("cons", "nunique"),
        broj_best_uses=("best_uses", "nunique"),
        broj_item_id=("item_id", "nunique"),
        broj_vrednosti_rating=("rating", "nunique"),
        broj_vrednosti_rating_count=("rating_count", "nunique"),
        broj_vrednosti_review_count=("review_count", "nunique"),
    )
    .reset_index()
)
duplirani_proizvodi=duplirani_proizvodi[duplirani_proizvodi.broj_redova>1]


In [76]:
display(pd.DataFrame([{
    "dupliranih logickih proizvoda": len(duplirani_proizvodi),
    "redova sa ponovljenim logickim proizvodima": duplirani_proizvodi.broj_redova.sum(),
    "visak redova": (duplirani_proizvodi.broj_redova - 1).sum(),
    "logickih proizvoda dvostruko ponovljenih": duplirani_proizvodi.broj_redova.eq(2).sum(),
    "logickih proizvoda trostruko ponovljenih": duplirani_proizvodi.broj_redova.eq(3).sum(),
}], index=['vrednost']).T)

,vrednost
dupliranih logickih proizvoda,96
redova sa ponovljenim logickim proizvodima,201
visak redova,105
logickih proizvoda dvostruko ponovljenih,87
logickih proizvoda trostruko ponovljenih,9


In [77]:
konflikti_unutar_log_proizv = pd.DataFrame([{
    "različit_product_name ": duplirani_proizvodi.broj_naziva.gt(1).sum(),
    "različit_description ": duplirani_proizvodi.broj_description.gt(1).sum(),
    "različit_price": duplirani_proizvodi.broj_cena.gt(1).sum(),
    "različit_rating_count ": duplirani_proizvodi.broj_vrednosti_rating_count.gt(1).sum(),
    "različit_brand": duplirani_proizvodi.broj_brendova.gt(1).sum(),
    "različit_category": duplirani_proizvodi.broj_kategorija.gt(1).sum(),
    "različit_item_id": duplirani_proizvodi.broj_item_id.gt(1).sum(),
    "različit_pros": duplirani_proizvodi.broj_pros.gt(1).sum(),
    "različit_cons": duplirani_proizvodi.broj_cons.gt(1).sum(),
    "različit_best_uses": duplirani_proizvodi.broj_best_uses.gt(1).sum(),
    "različit_rating": duplirani_proizvodi.broj_vrednosti_rating.gt(1).sum(),
    "različit_rating_count": duplirani_proizvodi.broj_vrednosti_rating_count.gt(1).sum(),
    "različit_review_count": duplirani_proizvodi.broj_vrednosti_review_count.gt(1).sum(),
}], index=["broj_grupa"]).T
konflikti_unutar_log_proizv[konflikti_unutar_log_proizv.broj_grupa.gt(0)]

,broj_grupa
različit_description,7
različit_brand,2
različit_item_id,9



### Zapažanje

Nakon uklanjanja **?sku=** iz URL-a:

- 1.373 reda predstavljaju 1.268 proizvoda
- 96 URL-ova pojavljuje se više puta
- oni obuhvataju 201 red
    - 87 proizvoda pojavljuje se dva puta
    - 9 proizvoda pojavljuje se tri puta
- postoji 105 dodatnih redova po logičkom proizvodu

Unutar tih 96 grupa:
- više razlicitih popunjenih vrednosti **brend** postoji u samo 2 grupe
- više razlicitih popunjenih vrednosti **category** postoji u samo 2 grupe
- 9 grupa ima više različitih **item_id** vrednosti


Provera da li se **item_id** moze naci iz  opisa i SKU
parametra iz URL-a.

In [78]:
proizvodi['item_id_iz_opisa'] = proizvodi.description.str.extract('\\bItem\\s+(\\d+)\\b', expand=False).astype(int)
proizvodi["item_id_iz_URLa"]=proizvodi.product_link.apply(lambda x: x.split('=')[1]).astype(int)
proizvodi["item_id"]=pd.to_numeric(proizvodi.item_id, errors='coerce')

In [79]:
display(pd.DataFrame(
    {
        "postoji_item_id": (proizvodi["item_id"].notna().sum()),
        "item_id_iz_opisa": (proizvodi["item_id_iz_opisa"].notna().sum()),
        "item_id_iz_URLa": (proizvodi["item_id_iz_URLa"].notna().sum()),
    },index=["broj_redova"],).T)

,broj_redova
postoji_item_id,853
item_id_iz_opisa,1373
item_id_iz_URLa,1373


In [80]:
proizvodi["existing_matches_description"] = proizvodi["item_id"]== proizvodi["item_id_iz_opisa"]
proizvodi["existing_matches_url"] = proizvodi["item_id"]== proizvodi["item_id_iz_URLa"]
proizvodi["description_matches_url"] = proizvodi["item_id_iz_opisa"]== proizvodi["item_id_iz_URLa"]

pregled_podudaranja_identifikatora = pd.Series(
    {
        "item_id = description_id": proizvodi["existing_matches_description"].sum(),
        "item_id = URL_id": proizvodi["existing_matches_url"].sum(),
        "description_id = URL_id": proizvodi["description_matches_url"].sum(),
    },
    name="broj_podudaranja",
).to_frame()
nepodudarni_identifikatori = proizvodi[~(proizvodi['existing_matches_description'].fillna(False) & proizvodi['existing_matches_url'].fillna(False))]

display(pregled_podudaranja_identifikatora)

,broj_podudaranja
item_id = description_id,850
item_id = URL_id,853
description_id = URL_id,1370


### Zapažanje
- item_id se moze najlakše oporaviti iz product_link
- item_id iz opisa i iz URL-a imaju potpunu pokrivenost i slažu se u svih 1.373 redova
- **item_id** nedostaje u 520 redova, ali se u svim popunjenim redovima slaže sa druga dva izvora

### Zaključak

- **product_link_id** -> Tehnički ID reda
- **item_id** -> SKU proizvoda
- **product_link**  -> URL konkretne varijante proizvoda  
- **logical_product_link** -> URL proizvoda bez SKU parametra 

# 3. Kvalitet atributa proizvoda

Proveravamo samo atribute koji određuju identitet, stroge uslove filtriranja, tekstualne dokumente ili statističko poreklo podataka.


## 3.1 Kategorije


In [81]:
logicki_proizvodi = proizvodi.groupby("logical_product_link")[["product_name", "category"]].first().reset_index()
display(logicki_proizvodi.category.value_counts(dropna=False).rename("broj_proizvoda").to_frame())

,broj_proizvoda
category,
Face Primer,241
Blush,190
Setting Spray & Powder,176
Foundation,153
Concealer,117
Bronzer,110
Highlighter,82
Makeup Remover,45
Contouring,45


In [82]:
cista_kategorija = proizvodi["category"].str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
pregled_kategorija = pd.DataFrame({
    "broj_kategorija": proizvodi.category.nunique(),
    "broj_cistih_kategorija": cista_kategorija.nunique(),
    "proizvodi_bez_kategorije": logicki_proizvodi.category.isna().sum(),
}, index=["vrednost"]).T
display(pregled_kategorija)

,vrednost
broj_kategorija,12
broj_cistih_kategorija,12
proizvodi_bez_kategorije,16


In [83]:
proizvodi[proizvodi.category.isna()][['product_name','category']]

,product_name,category
0,Futurist Skin Tint Serum Foundation SPF 20,NaN
1,Dior Forever Fluid Skin Glow Foundation,NaN
2,BAREPRO 24HR Wear Skin-Perfecting Matte Liquid Foundation Mineral SPF 20,NaN
3,Futurist Hydra Rescue Moisturizing Foundation SPF 45,NaN
154,Trick and Treat CC² Active Propolis Color Correcting Cream With Broad Spectrum SPF 45,NaN
155,COMPLEXION RESCUE Natural Matte Tinted Moisturizer Mineral SPF 30,NaN
156,Tinted Moisturizer Oil Free Natural Skin Perfector Broad Spectrum SPF 20,NaN
162,Mini Halo Healthy Glow Tinted Moisturizer Broad Spectrum SPF 25,NaN
163,Mini Tinted Moisturizer Natural Skin Perfector Broad Spectrum SPF 30,NaN
164,Mini Tinted Moisturizer Oil Free Natural Skin Perfector Broad Spectrum SPF 20,NaN


In [84]:
visestruka_katagorija=duplirani_proizvodi[duplirani_proizvodi.broj_kategorija.gt(1)].logical_product_link.values.tolist()
proizvodi[proizvodi.logical_product_link.isin(visestruka_katagorija)][['product_name','category']]

,product_name,category


### Zaključak
- kategorija nedostaje za 16 od 1.268 logičkih proizvoda.

## 3.2 Brend


In [85]:
visestruki_brand=duplirani_proizvodi[duplirani_proizvodi.broj_brendova.gt(1)].logical_product_link.values.tolist()
proizvodi[proizvodi.logical_product_link.isin(visestruki_brand)][['product_name','brand']].sort_values(by='product_name')

,product_name,brand
0,Futurist Skin Tint Serum Foundation SPF 20,Find your shade
77,Futurist Skin Tint Serum Foundation SPF 20,Estée Lauder
4,Mini CC+ Cream with SPF 50+,IT Cosmetics
157,Mini CC+ Cream with SPF 50+,Find your shade


In [86]:
logicki_proizvodi = proizvodi.groupby("logical_product_link")[["product_name", "brand"]].first().reset_index()
display(logicki_proizvodi.brand.value_counts(dropna=False).rename("broj_proizvoda").to_frame())

,broj_proizvoda
brand,
NaN,409
Ask A Question,215
Find your shade,56
Tarte,39
e.l.f. Cosmetics,21
...,...
KIKO Milano,1
Exa,1
Write A Review,1


In [ ]:
neispravni_brendovi = {"Ask A Question","Find your shade","Write A Review"}
proizvodi["brand"] = proizvodi.brand.str.strip().replace("", pd.NA).where(~proizvodi["brand"].isin(neispravni_brendovi))
cisti_brendovi = proizvodi.groupby("logical_product_link", as_index=False).agg(
        brand_clean=("brand",lambda x: x.dropna().iloc[0]if x.notna().any() else pd.NA))

mapa_brendova = dict(zip(cisti_brendovi["logical_product_link"],cisti_brendovi["brand_clean"]))
proizvodi["brand"] = proizvodi["logical_product_link"].map(mapa_brendova)

In [ ]:
#nadji poziciju naziva proizvoda i izvuci sta pise pre
proizvodi["zaglavlje_opisa"] = proizvodi["description"].str.split("Summary").str[0].str.replace(
    r"(?is)^.*?Tab through the images or use the previous or next buttons to navigate each product image", ""
    , regex=True).str.replace(
    r"(?i)^\s*try it\s*", "", regex=True).str.strip()

def izdvoji_brend(red):
    pozicija = red.zaglavlje_opisa.find(str(red.product_name))
    return red.zaglavlje_opisa[:pozicija].strip() if pozicija > 0 else pd.NA

proizvodi["brand_iz_opisa"] = proizvodi.apply(izdvoji_brend, axis=1)

display(pd.DataFrame([
    {'poznato brandova':pd.notna(proizvodi.brand).sum(),
    'podudaranja sa izvucenim iz opisa':  (proizvodi["brand"].eq(proizvodi["brand_iz_opisa"])).sum(), 
    'nedostaje brand iz opisa': pd.isna(proizvodi.brand_iz_opisa).sum()}
], index=['Broj redova']).T)
print('Nedostaju vrednosti za: ')
proizvodi[proizvodi.brand_iz_opisa.isna()][['product_name']]


,Broj redova
poznato brandova,646
podudaranja sa izvucenim iz opisa,646
nedostaje brand iz opisa,2


Nedostaju vrednosti za: 


,product_name
92,Born This Way Soft Matte Foundation
239,ORIGINAL Liquid Mineral Concealer


In [89]:
#rucno dopuni za nedostajuce
proizvodi["brand_konacno"] = proizvodi["brand_iz_opisa"]
for naziv, brend in {"Born This Way Soft Matte Foundation": "Too Faced", "ORIGINAL Liquid Mineral Concealer": "bareMinerals"}.items():
    proizvodi.loc[proizvodi["product_name"].str.contains(naziv) & proizvodi["brand_konacno"].isna(), "brand_konacno"] = brend


In [90]:
display(pd.DataFrame([
    {'poznato brandova':pd.notna(proizvodi.brand).sum(),
    'podudaranja sa izvucenim iz opisa':  (proizvodi["brand"].eq(proizvodi["brand_iz_opisa"])).sum(), 
    'nedostaje brand iz opisa': pd.isna(proizvodi.brand_konacno).sum()}
], index=['Broj redova']).T)


,Broj redova
poznato brandova,646
podudaranja sa izvucenim iz opisa,646
nedostaje brand iz opisa,0


### Zaključak

- brend sadrži nedostajuće i nevalidne vrednosti .
- nema konflikta brenda iz opisa i poznatih brendova po logičkom proizvodu.
- potrebno dopuniti nedostajuće vrednosti za 2 proizvoda

## 3.3 Cena


In [91]:
proizvodi["price"] = pd.to_numeric(proizvodi["price"], errors="coerce")
pregled_cena = pd.Series({
    "nedostajuce_ili_neispravne": proizvodi["price"].isna().sum(),
    "nulte": proizvodi["price"].eq(0).sum(),
    "negativne": proizvodi["price"].lt(0).sum(),
    "najmanja_cena": proizvodi["price"].min(),
    "najveca_cena": proizvodi["price"].max(),
}, name="vrednost").to_frame()
display(pregled_cena)


,vrednost
nedostajuce_ili_neispravne,0.0
nulte,0.0
negativne,0.0
najmanja_cena,1.5
najveca_cena,85.0


### Zaključak

- Sve cene su popunjene, i vece od nule.
- Nema sukoba cene u okviru proizvoda.


## 3.4 Ocene i broj recenzija

Razdvajamo vrednosti prikazane u zaglavlju stranice, platformske agregate ocena i recenzija i izvedene agregate iz dostupnog skupa recenzija.


In [92]:
opis_tekst = proizvodi["description"].astype("string").str.split("Summary", n=1).str[0]
ocena_iz_opisa = pd.to_numeric(opis_tekst.str.extract(r"Item\s+\d+\s+(\d+(?:\.\d+)?)\s+\d+(?:\.\d+)?\s+out of 5 stars", expand=False), errors="coerce")
recenzije_iz_opisa = pd.to_numeric(opis_tekst.str.extract(r"out of 5 stars\.\s*([\d,]+)\s+reviews", expand=False).str.replace(",", "", regex=False), errors="coerce")

kolone_zvezdica_rating = [f"rating_star_{i}" for i in range(1, 6)]
kolone_zvezdica_review = [f"review_star_{i}" for i in range(1, 6)]

zbir_rating_star = proizvodi[kolone_zvezdica_rating].sum(axis=1)
zbir_review_star = proizvodi[kolone_zvezdica_review].sum(axis=1)

ponderisana_suma = sum(proizvodi[f"rating_star_{i}"] * i for i in range(1, 6))
izracunata_ocena = (ponderisana_suma / zbir_rating_star).round(2)

num_cols = ["rating", "average_rating", "num_reviews", "rating_count", "review_count"]
num_df = proizvodi[num_cols].apply(pd.to_numeric, errors="coerce")

provere = {
    "count(rating_star) != rating_count": zbir_rating_star.ne(num_df["rating_count"]).sum(),
    "count(review_star) != review_count": zbir_review_star.ne(num_df["review_count"]).sum(),
    "rating != opis_rating" :  num_df.rating[num_df.rating.notna() & ocena_iz_opisa.notna()].ne(ocena_iz_opisa[num_df.rating.notna() & ocena_iz_opisa.notna()]).sum()
}

for kolona in ["num_reviews", "rating_count", "review_count"]:
    uporedivo = num_df[kolona].notna() & recenzije_iz_opisa.notna()
    provere[f"{kolona} != opis_{kolona}"] = num_df[kolona][uporedivo].ne(recenzije_iz_opisa[uporedivo]).sum()

# Prikaz rezultata
display(pd.Series(provere, name="broj_redova").to_frame())

,broj_redova
count(rating_star) != rating_count,0
count(review_star) != review_count,0
rating != opis_rating,0
num_reviews != opis_num_reviews,0
rating_count != opis_rating_count,222
review_count != opis_review_count,211


### Zapažanje
- rating_count i review_count su tačni na skupu podataka
- rating i num_reviews se poklapaju sa vrednostima iz opisa, predstavljaju vrednosti u trenutku prikupljanja podataka


## 3.5 Tekstualna polja i opis proizvoda

In [93]:
import re
import pandas as pd

def pronadji_pocetak_sastojaka(tekst):
    posebni_format = re.search(
        r"""(?ix)
        \b(
            Ingredients\s+Active\s+Ingredients |
            Ingredients\s*/\s*Ingredients\s+Active |
            Ingredients\s+Active |
            Active\s+Ingredients
        )\s*:
        """,
        tekst
    )
    if posebni_format:
        return posebni_format.start()
    kandidati = list(re.finditer(r"(?i)\bIngredients\b\s*:?",tekst))
    for kandidat in reversed(kandidati):
        tekst_pre = tekst[max(0, kandidat.start() - 20):kandidat.start()]
        if re.search(r"(?i)\b(?:Active|Inactive)\s*$", tekst_pre):
            continue
        nastavak = tekst[kandidat.end():].strip()
        if re.match(r"(?i)^(may|are|can|could|should|subject|vary|change)\b",nastavak):
            continue
        if "," in nastavak[:500] or "%" in nastavak[:500]:
            return kandidat.start()
    return None

def razdvoji_opis_i_sastojke(opis):
    if pd.isna(opis):
        return pd.Series({"description_clean": pd.NA,"ingredients": pd.NA})
    summary = re.search(r"(?is)\bSummary\b\s*(.*)",str(opis))
    if summary is None:
        return pd.Series({"description_clean": pd.NA,"ingredients": pd.NA})
    tekst = summary.group(1)
    tekst = re.split(r"(?is)\bShipping\s*&\s*Coupon Restrictions\b",tekst,maxsplit=1)[0]
    pocetak = pronadji_pocetak_sastojaka(tekst)
    if pocetak is None:
        description_clean = tekst
        ingredients = pd.NA
    else:
        description_clean = tekst[:pocetak]
        ingredients = tekst[pocetak:]
    description_clean = re.sub(r"\s+"," ",description_clean).strip()
    if pd.notna(ingredients):
        ingredients = re.sub(r"\s+"," ",ingredients).strip()
    return pd.Series({
        "description_clean": description_clean or pd.NA,
        "ingredients": ingredients if pd.notna(ingredients) else pd.NA
    })

In [94]:
proizvodi[["description_clean", "ingredients"]] = proizvodi["description"].apply(razdvoji_opis_i_sastojke)

In [95]:
konflikti = proizvodi.groupby("logical_product_link", dropna=False).agg(
    broj_opisa=("description_clean","nunique"),
    broj_sastojaka=("ingredients","nunique"),
)

display( pd.DataFrame([
    {
        "neuspesno_izdvajanje_opisa":proizvodi["description_clean"].isna().sum(),
        "prazan_izdvojen_opis":proizvodi["description_clean"].fillna("").str.strip().eq("").sum(),
        "proizvodi_sa_vise_opisa":konflikti.broj_opisa.gt(1).sum(),
        "proizvodi_sa_vise_lista_sastojaka":konflikti.broj_sastojaka.gt(1).sum(),
    }],
    index=["broj_proizvoda"]).T)


,broj_proizvoda
neuspesno_izdvajanje_opisa,0
prazan_izdvojen_opis,0
proizvodi_sa_vise_opisa,0
proizvodi_sa_vise_lista_sastojaka,0
